In [ ]:
class CyberExpertSystem:
    def __init__(self):
        self.rules = [
            {'conditions': ['high_traffic', 'many_sources', 'no_user_activity'], 'conclusion': 'ddos',
             'desc': 'High traffic from many sources with no user activity indicates DDoS.'},
            {'conditions': ['high_traffic', 'single_source'], 'conclusion': 'ddos',
             'desc': 'High traffic from single source also suggests DDoS flood.'},
            {'conditions': ['many_ports', 'short_time', 'low_traffic'], 'conclusion': 'port_scan',
             'desc': 'Connections to many ports in short time with low traffic per connection signals port scan.'},
            {'conditions': ['many_ports', 'sequential_ports'], 'conclusion': 'port_scan',
             'desc': 'Sequential port probes confirm scanning activity.'},
            {'conditions': ['suspicious_payloads', 'beaconing'], 'conclusion': 'malware',
             'desc': 'Suspicious payloads with periodic beaconing indicate malware C2.'},
            {'conditions': ['suspicious_payloads', 'high_entropy'], 'conclusion': 'malware',
             'desc': 'High-entropy payloads suggest packed malware.'},
        ]
        self.facts = {}
        self.trace = []

    def ask_user(self, fact):
        """Query user for missing fact, like a diagnostic tool."""
        response = input(f"Is '{fact}' true in the logs? (y/n): ").lower().strip()
        return response in ('y', 'yes', 'true', '1')

    def forward_chain(self):
        """Perform forward chaining until no new facts are derived."""
        self.trace = []
        changed = True
        while changed:
            changed = False
            for rule in self.rules:
                if all(self.facts.get(cond, False) for cond in rule['conditions']):
                    if not self.facts.get(rule['conclusion'], False):
                        self.facts[rule['conclusion']] = True
                        changed = True
                        self.trace.append(f"FIRED: {rule['desc']}")
        return self.diagnose()

    def diagnose(self):
        """Determine primary threat from derived facts."""
        threats = [t for t in ['ddos', 'port_scan', 'malware'] if self.facts.get(t, False)]
        return threats[0] if threats else 'normal'

test_scenarios = [
    {'high_traffic': True, 'many_sources': True, 'no_user_activity': True},
    {'high_traffic': True, 'single_source': True},
    {'high_traffic': True, 'many_sources': True},
    {'many_ports': True, 'short_time': True, 'low_traffic': True},
    {'many_ports': True, 'sequential_ports': True},
    {'many_ports': True, 'short_time': True},
    {'suspicious_payloads': True, 'beaconing': True},
    {'suspicious_payloads': True, 'high_entropy': True},
    {'suspicious_payloads': True, 'beaconing': True, 'high_traffic': True},  
    {'suspicious_payloads': True, 'high_entropy': True},
]

def run_tests():
    results = []
    for i, scenario in enumerate(test_scenarios, 1):
        sys = CyberExpertSystem()
        sys.facts = scenario.copy()
        expected = 'ddos' if i <= 3 else 'port_scan' if i <= 6 else 'malware'
        diagnosed = sys.forward_chain()
        trace = sys.trace
        correct = diagnosed == expected
        results.append({'scenario': i, 'facts': scenario, 'diagnosed': diagnosed, 'expected': expected, 'correct': correct, 'trace': trace})
        print(f"\n--- Scenario {i} ---")
        print(f"Input facts: {scenario}")
        print(f"Diagnosed: {diagnosed} (Expected: {expected}) -> {'PASS' if correct else 'FAIL'}")
        print("Reasoning trace:")
        for t in trace:
            print(f"  {t}")

    correct = sum(r['correct'] for r in results)
    print(f"\nOverall Accuracy: {correct}/10 = {correct*10}%")

def interactive_mode():
    sys = CyberExpertSystem()
    print("Provide log facts (or run tests first).")
    all_facts = set(c for r in sys.rules for c in r['conditions'])
    for fact in all_facts:
        if fact not in sys.facts:
            sys.facts[fact] = sys.ask_user(fact)
    diagnosis = sys.forward_chain()
    print(f"\nDiagnosis: {diagnosis}")
    print("Reasoning trace:")
    for step in sys.trace:
        print(f"- {step}")

if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == 'test':
        run_tests()
    else:
        print("Run with 'test' arg for automated tests, or interactively.")
        interactive_mode()


Run with 'test' arg for automated tests, or interactively.
Provide log facts (or run tests first).

Diagnosis: ddos
Reasoning trace:
- FIRED: High traffic from many sources with no user activity indicates DDoS.
- FIRED: Connections to many ports in short time with low traffic per connection signals port scan.
- FIRED: Suspicious payloads with periodic beaconing indicate malware C2.
